# Kombinieren und Zusammenführen von Datensätzen

Daten, die in pandas-Objekten enthalten sind, können auf verschiedene Weise kombiniert werden:

* [pandas.merge](https://pandas.pydata.org/docs/reference/api/pandas.merge.html) führt Zeilen in DataFrames anhand eines oder mehrerer Schlüssel zusammen. Diese Funktion ist aus SQL oder anderen relationalen Datenbanken bekannt, da sie Datenbank-Join-Operationen implementiert.
* [pandas.concat](https://pandas.pydata.org/docs/reference/api/pandas.concat.html) verkettet oder stapelt Objekte entlang einer Achse.
* Die Instanzmethoden [pandas.DataFrame.combine_first](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.combine_first.html) oder [pandas.Series.combine_first](https://pandas.pydata.org/docs/reference/api/pandas.Series.combine_first.html) ermöglichen das Verknüpfen überlappender Daten.
* [pandas.merge_asof](https://pandas.pydata.org/docs/reference/api/pandas.merge_asof.html) führt eine Verknüpfung nach Schlüsselabstand durch.

## Datenbankähnliche DataFrame-Verknüpfungen

Merge- oder Join-Operationen kombinieren Datensätze, indem sie Zeilen anhand eines oder mehrerer Schlüssel verknüpfen. Diese Operationen sind besonders wichtig in relationalen, SQL-basierten Datenbanken. Die Merge-Funktion in pandas ist der zentrale Einstiegspunkt für die Anwendung dieser Algorithmen auf Ihre Daten.

In [1]:
import pandas as pd

In [2]:
encoding = pd.DataFrame(
    {
        "Unicode": [
            "U+0000",
            "U+0001",
            "U+0002",
            "U+0003",
            "U+0004",
            "U+0005",
        ],
        "Decimal": [0, 1, 2, 3, 4, 5],
        "Octal": ["000", "001", "002", "003", "004", "005"],
        "Key": ["NUL", "Ctrl-A", "Ctrl-B", "Ctrl-C", "Ctrl-D", "Ctrl-E"],
    },
)

update = pd.DataFrame(
    {
        "Unicode": [
            "U+0003",
            "U+0004",
            "U+0005",
            "U+0006",
            "U+0007",
            "U+0008",
            "U+0009",
        ],
        "Decimal": [3, 4, 5, 6, 7, 8, 9],
        "Octal": ["003", "004", "005", "006", "007", "008", "009"],
        "Key": [
            "Ctrl-C",
            "Ctrl-D",
            "Ctrl-E",
            "Ctrl-F",
            "Ctrl-G",
            "Ctrl-H",
            "Ctrl-I",
        ],
    },
)

In [3]:
encoding

,Unicode,Decimal,Octal,Key
0,U+0000,0,000,NUL
1,U+0001,1,001,Ctrl-A
2,U+0002,2,002,Ctrl-B
3,U+0003,3,003,Ctrl-C
4,U+0004,4,004,Ctrl-D
5,U+0005,5,005,Ctrl-E


In [4]:
update

,Unicode,Decimal,Octal,Key
0,U+0003,3,003,Ctrl-C
1,U+0004,4,004,Ctrl-D
2,U+0005,5,005,Ctrl-E
3,U+0006,6,006,Ctrl-F
4,U+0007,7,007,Ctrl-G
5,U+0008,8,008,Ctrl-H
6,U+0009,9,009,Ctrl-I


Wenn wir `df.merge()` aufrufen, erhalten wir:

In [5]:
encoding.merge(update)

,Unicode,Decimal,Octal,Key
0,U+0003,3,003,Ctrl-C
1,U+0004,4,004,Ctrl-D
2,U+0005,5,005,Ctrl-E


Standardmäßig führt `merge` einen sogenannten *Inner Join* durch; die Schlüssel im Ergebnis sind die Schnittmenge oder die gemeinsamen Elemente beider Tabellen.

<div class="alert alert-block alert-info">

**Hinweis:**

Ich habe nicht angegeben, über welcher Spalte die Zusammenführung erfolgen soll. Wenn diese Information nicht angegeben wird, verwendet `merge` die übereinstimmenden Spaltennamen als Schlüssel. Es empfiehlt sich jedoch, dies explizit anzugeben:
</div>

In [6]:
encoding.merge(update, on="Unicode")

,Unicode,Decimal_x,Octal_x,Key_x,Decimal_y,Octal_y,Key_y
0,U+0003,3,003,Ctrl-C,3,003,Ctrl-C
1,U+0004,4,004,Ctrl-D,4,004,Ctrl-D
2,U+0005,5,005,Ctrl-E,5,005,Ctrl-E


Wenn die Spaltennamen in den einzelnen Objekten unterschiedlich sind, könnt ihr sie separat angeben. Im folgenden Beispiel erhält `update2` den Schlüssel `U+` und nicht `Unicode`:

In [7]:
update2 = pd.DataFrame(
    {
        "U+": [
            "U+0003",
            "U+0004",
            "U+0005",
            "U+0006",
            "U+0007",
            "U+0008",
            "U+0009",
        ],
        "Decimal": [3, 4, 5, 6, 7, 8, 9],
        "Octal": ["003", "004", "005", "006", "007", "008", "009"],
        "Key": [
            "Ctrl-C",
            "Ctrl-D",
            "Ctrl-E",
            "Ctrl-F",
            "Ctrl-G",
            "Ctrl-H",
            "Ctrl-I",
        ],
    },
)

encoding.merge(update2, left_on="Unicode", right_on="U+")

,Unicode,Decimal_x,Octal_x,Key_x,U+,Decimal_y,Octal_y,Key_y
0,U+0003,3,003,Ctrl-C,U+0003,3,003,Ctrl-C
1,U+0004,4,004,Ctrl-D,U+0004,4,004,Ctrl-D
2,U+0005,5,005,Ctrl-E,U+0005,5,005,Ctrl-E


Ihr könnt `.merge` jedoch nicht nur für einen Inner Join verwenden, bei dem die Schlüssel im Ergebnis die Schnittmenge oder die gemeinsamen Elemente beider Tabellen sind. Weitere mögliche Optionen sind:

Option | Verhalten
:----- | :--------
`how='inner'` | verwendet nur die Schlüsselkombinationen, die in beiden Tabellen vorkommen
`how='left'` | verwendet alle Schlüsselkombinationen, die in der linken Tabelle vorkommen
`how='right'` | verwendet alle Schlüsselkombinationen, die in der rechten Tabelle vorkommen
`how='outer'` | verwendet alle Schlüsselkombinationen, die in beiden Tabellen vorkommen

In [8]:
encoding.merge(update, on="Unicode", how="left")

,Unicode,Decimal_x,Octal_x,Key_x,Decimal_y,Octal_y,Key_y
0,U+0000,0,000,NUL,NaN,NaN,NaN
1,U+0001,1,001,Ctrl-A,NaN,NaN,NaN
2,U+0002,2,002,Ctrl-B,NaN,NaN,NaN
3,U+0003,3,003,Ctrl-C,3.0,003,Ctrl-C
4,U+0004,4,004,Ctrl-D,4.0,004,Ctrl-D
5,U+0005,5,005,Ctrl-E,5.0,005,Ctrl-E


In [9]:
encoding.merge(update, on="Unicode", how="outer")

,Unicode,Decimal_x,Octal_x,Key_x,Decimal_y,Octal_y,Key_y
0,U+0000,0.0,000,NUL,NaN,NaN,NaN
1,U+0001,1.0,001,Ctrl-A,NaN,NaN,NaN
2,U+0002,2.0,002,Ctrl-B,NaN,NaN,NaN
3,U+0003,3.0,003,Ctrl-C,3.0,003,Ctrl-C
4,U+0004,4.0,004,Ctrl-D,4.0,004,Ctrl-D
5,U+0005,5.0,005,Ctrl-E,5.0,005,Ctrl-E
6,U+0006,NaN,NaN,NaN,6.0,006,Ctrl-F
7,U+0007,NaN,NaN,NaN,7.0,007,Ctrl-G
8,U+0008,NaN,NaN,NaN,8.0,008,Ctrl-H
9,U+0009,NaN,NaN,NaN,9.0,009,Ctrl-I


Die Join-Methode wirkt sich nur auf die eindeutigen Schlüsselwerte aus, die im Ergebnis erscheinen.

Um mehrere Schlüssel zu verknüpfen, könnt ihr eine Liste mit Spaltennamen übergeben:

In [10]:
encoding.merge(update, on=["Unicode", "Decimal", "Octal", "Key"], how="outer")

,Unicode,Decimal,Octal,Key
0,U+0000,0,000,NUL
1,U+0001,1,001,Ctrl-A
2,U+0002,2,002,Ctrl-B
3,U+0003,3,003,Ctrl-C
4,U+0004,4,004,Ctrl-D
5,U+0005,5,005,Ctrl-E
6,U+0006,6,006,Ctrl-F
7,U+0007,7,007,Ctrl-G
8,U+0008,8,008,Ctrl-H
9,U+0009,9,009,Ctrl-I
